In [ ]:
import pandas as pd
from plotnine import *
from vpop_calibration import *

%load_ext autoreload
%autoreload 2

In [ ]:
df = pd.read_csv("nimoData.csv")

doses = df[df["EVID"] == 1]

doses_wide = doses.pivot(index="ID", columns="OCC", values=["TIME", "AMT"])

doses_wide.columns = [
    f"time{int(occ)}" if col == "TIME" else f"dose{int(occ)}"
    for col, occ in doses_wide.columns
]

doses_wide = doses_wide.reset_index().rename(columns={"ID": "id"})

obs_df = (
    df.loc[df["EVID"] == 0]
    .rename(columns={"ID": "id", "DV": "value", "TIME": "time"})[
        ["id", "time", "value"]
    ]
    .astype({"value": "float"})
)

obs_df["time"] = obs_df["time"] * 3600
obs_df = obs_df[obs_df["value"] > 0]
# obs_df["value"] = obs_df["value"].apply(lambda t: np.log(t))
obs_df["output_name"] = "IPRED"
obs_df["protocol_arm"] = "identity"

obs_df = obs_df.merge(doses_wide, on="id", how="left").fillna(0.0)

display(obs_df.head())

In [ ]:
model = SimworkModelBinding(
    path_to_model="CM_Nimotuzumab.json",
    path_to_solving_options="SV_Nimotuzumab.json",
    inputs=[
        "cl",
        "v1",
        "Q",
        "v2",
        "k_ss",
        "k_int",
        "k_syn",
        "k_deg",
        "time1",
        "time2",
        "time3",
        "time4",
        "time5",
        "time6",
        "time7",
        "time8",
        "time9",
        "time10",
        "dose1",
        "dose2",
        "dose3",
        "dose4",
        "dose5",
        "dose6",
        "dose7",
        "dose8",
        "dose9",
        "dose10",
    ],
    outputs=["IPRED"],
)
print(model.inputs)

struct_model = StructuralSimwork(model=model)

In [ ]:
prior_pdu = {
    "model_intrinsic": {
        "Q": {"prior": 0.004},
        "v2": {"prior": 44},
        "k_int": {"prior": 0.3},
        "k_syn": {"prior": 1},
        "k_deg": {"prior": 7},
    },
    "pdu": {
        "cl": {"prior": 0.001, "prior_omega": 2},
        "v1": {"prior": 1.45, "prior_omega": 2},
        "k_ss": {"prior": 12, "prior_omega": 2},
    },
    "pdk": {
        "time1",
        "time2",
        "time3",
        "time4",
        "time5",
        "time6",
        "time7",
        "time8",
        "time9",
        "time10",
        "dose1",
        "dose2",
        "dose3",
        "dose4",
        "dose5",
        "dose6",
        "dose7",
        "dose8",
        "dose9",
        "dose10",
    },
    "error_model": {
        "IPRED": {"error_type": "additive", "sigma": 2},
    },
}

config = Config(
    saem=SaemConfigDict(
        nb_iter_burnin=0,
        nb_iter_learning=30,
        nb_iter_smoothing=30,
        plot_frames=5,
        optim_max_fun=20,
    ),
    nlme=NlmeConfigDict(nb_chains=1),
)

nlme_model = NlmeModel(
    df=obs_df, prior_params=prior_pdu, structural_model=struct_model, config=config
)

In [ ]:
nlme_model.optimizer.run()

In [ ]:
nlme_model.diagnostics.sample_conditional_distribution(200)

In [ ]:
nlme_model.plot.map_estimates_gof()

In [ ]:
nlme_model.plot.map_estimates()

In [ ]:
nlme_model.plot.weighted_residuals("iwres")

In [ ]:
nlme_model.plot.weighted_residuals("pwres")